# Project Nano-Myo
## Notebook 02 - Feature Extraction


## Step 1 - Setup


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Step 2 - Configuration


In [ ]:
import json
from collections import Counter
from pathlib import Path

import numpy as np
import scipy.io
from sklearn.preprocessing import StandardScaler

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

DRIVE_ROOT = Path('/content/drive/MyDrive')
PROJECT_ROOT = DRIVE_ROOT / 'Nano_Myo'
EXTRACTED_DIR = PROJECT_ROOT / 'extracted'
FEATURE_DIR = PROJECT_ROOT / 'features'
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_SCHEMA_VERSION = 'e2_only_80feat_wl_ssc_rest25000_v1'
SAMPLE_RATE_HZ = 200
WINDOW_MS = 150
OVERLAP = 0.75
WINDOW_SIZE = int(round(SAMPLE_RATE_HZ * WINDOW_MS / 1000))
STEP_SIZE = max(1, int(round(WINDOW_SIZE * (1 - OVERLAP))))
PURITY_THRESHOLD = 0.75
ZC_THRESHOLD = 0.01
SSC_THRESHOLD = 0.0001
REST_CAP = 25000

TRAIN_SUBJECTS = [f's{i}' for i in range(1, 9)]
TEST_SUBJECTS = ['s9', 's10']
ALL_SUBJECTS = TRAIN_SUBJECTS + TEST_SUBJECTS

E2_LABEL_MAP = {
    0: 0,
    1: 1,
    2: 2,
    4: 3,
    5: 4,
    9: 5,
    10: 6,
    11: 7,
    12: 8,
    3: 9,
}

TARGET_LABELS = {
    0: 'Rest',
    1: 'Hand open',
    2: 'Hand close',
    3: 'Pinch',
    4: 'Tripod grasp',
    5: 'Wrist flexion',
    6: 'Wrist extension',
    7: 'Wrist pronation',
    8: 'Wrist supination',
    9: 'Lateral grasp',
}

FEATURE_GROUPS = ['mav', 'zc', 'rms', 'wl', 'ssc']
FEATURE_NAMES = [f'{group}_ch{channel:02d}' for group in FEATURE_GROUPS for channel in range(16)]

assert WINDOW_SIZE == 30
assert STEP_SIZE == 8
assert len(FEATURE_NAMES) == 80
assert E2_LABEL_MAP[4] == 3
assert E2_LABEL_MAP[5] == 4


## Step 3 - Discover Files


In [ ]:
def is_e2_file(path):
    name = path.stem.lower().replace('-', '_')
    return '_e2_' in name or name.endswith('_e2') or 'e2' in name


def discover_e2_files(subject_id):
    subject_dir = EXTRACTED_DIR / subject_id
    if not subject_dir.exists():
        raise FileNotFoundError(f'Missing extracted subject directory: {subject_dir}')
    files = sorted(path for path in subject_dir.rglob('*.mat') if is_e2_file(path))
    if not files:
        raise FileNotFoundError(f'No E2 .mat files found for {subject_id}: {subject_dir}')
    return files


e2_files = {subject: discover_e2_files(subject) for subject in ALL_SUBJECTS}

for subject, files in e2_files.items():
    print(f'{subject}: {len(files)} E2 file(s)')
    for path in files:
        print(f'  {path.name}')


## Step 4 - Helpers


In [ ]:
def mat_key(mat, candidates):
    lower_keys = {key.lower(): key for key in mat.keys()}
    for candidate in candidates:
        key = lower_keys.get(candidate.lower())
        if key is not None:
            return key
    raise KeyError(f'None of these keys were found: {candidates}')


def majority_label(labels):
    values, counts = np.unique(labels.astype(np.int32), return_counts=True)
    return int(values[np.argmax(counts)])


def window_features(window):
    window = np.asarray(window, dtype=np.float32)
    diff = np.diff(window, axis=0)
    mav = np.mean(np.abs(window), axis=0)
    zc = np.sum((np.signbit(window[:-1]) != np.signbit(window[1:])) & (np.abs(diff) >= ZC_THRESHOLD), axis=0)
    rms = np.sqrt(np.mean(np.square(window), axis=0))
    wl = np.sum(np.abs(diff), axis=0)
    prev_slope = window[1:-1] - window[:-2]
    next_slope = window[1:-1] - window[2:]
    ssc = np.sum((prev_slope * next_slope) >= SSC_THRESHOLD, axis=0)
    return np.concatenate([mav, zc, rms, wl, ssc]).astype(np.float32)


def rest_hardness_score(window):
    window = np.asarray(window, dtype=np.float32)
    mav = np.mean(np.abs(window), axis=0)
    wl = np.sum(np.abs(np.diff(window, axis=0)), axis=0)
    return float(np.mean(mav) + 0.25 * np.mean(wl))


def extract_features_from_file(path, subject_id):
    mat = scipy.io.loadmat(path)
    emg_key = mat_key(mat, ['emg'])
    label_key = mat_key(mat, ['restimulus', 'stimulus'])
    emg = np.asarray(mat[emg_key], dtype=np.float32)
    labels = np.asarray(mat[label_key]).reshape(-1).astype(np.int32)

    if emg.ndim != 2:
        raise ValueError(f'Expected 2D EMG array in {path}')
    if emg.shape[1] < 16:
        raise ValueError(f'Expected at least 16 EMG channels in {path}, found {emg.shape[1]}')

    length = min(emg.shape[0], labels.shape[0])
    emg = emg[:length, :16]
    labels = labels[:length]

    X, y, source_labels, rest_scores = [], [], [], []
    for start in range(0, length - WINDOW_SIZE + 1, STEP_SIZE):
        stop = start + WINDOW_SIZE
        label_window = labels[start:stop]
        raw_label = majority_label(label_window)
        if raw_label not in E2_LABEL_MAP:
            continue
        if np.mean(label_window == raw_label) < PURITY_THRESHOLD:
            continue
        emg_window = emg[start:stop]
        X.append(window_features(emg_window))
        y.append(E2_LABEL_MAP[raw_label])
        source_labels.append(raw_label)
        rest_scores.append(rest_hardness_score(emg_window) if raw_label == 0 else 0.0)

    if not X:
        return (
            np.empty((0, len(FEATURE_NAMES)), dtype=np.float32),
            np.empty((0,), dtype=np.int64),
            np.empty((0,), dtype=np.int64),
            np.empty((0,), dtype=np.float32),
        )

    return (
        np.vstack(X).astype(np.float32),
        np.asarray(y, dtype=np.int64),
        np.asarray(source_labels, dtype=np.int64),
        np.asarray(rest_scores, dtype=np.float32),
    )


## Step 5 - Extract Features


In [ ]:
all_X, all_y, all_subjects, all_source_labels, all_rest_scores = [], [], [], [], []

for subject in ALL_SUBJECTS:
    subject_windows = 0
    for path in e2_files[subject]:
        X_part, y_part, source_part, rest_part = extract_features_from_file(path, subject)
        if X_part.shape[0] == 0:
            continue
        all_X.append(X_part)
        all_y.append(y_part)
        all_subjects.append(np.full(y_part.shape[0], subject, dtype=object))
        all_source_labels.append(source_part)
        all_rest_scores.append(rest_part)
        subject_windows += y_part.shape[0]
    print(f'{subject}: {subject_windows:,} windows')

X_all = np.vstack(all_X).astype(np.float32)
y_all = np.concatenate(all_y).astype(np.int64)
subjects_all = np.concatenate(all_subjects)
source_labels_all = np.concatenate(all_source_labels).astype(np.int64)
rest_scores_all = np.concatenate(all_rest_scores).astype(np.float32)

if X_all.shape[1] != 80:
    raise ValueError(f'Expected 80 features, found {X_all.shape[1]}')
if set(np.unique(y_all)) != set(TARGET_LABELS.keys()):
    raise ValueError(f'Unexpected class set: {sorted(np.unique(y_all).tolist())}')

print(f'Total windows: {X_all.shape[0]:,}')
print(f'Feature shape: {X_all.shape}')
print(Counter(y_all.tolist()))


## Step 6 - Split And Normalize


In [ ]:
train_mask = np.isin(subjects_all, TRAIN_SUBJECTS)
test_mask = np.isin(subjects_all, TEST_SUBJECTS)

X_train_raw = X_all[train_mask]
y_train_raw = y_all[train_mask]
subjects_train_raw = subjects_all[train_mask]
source_labels_train_raw = source_labels_all[train_mask]
rest_scores_train_raw = rest_scores_all[train_mask]

X_test_raw = X_all[test_mask]
y_test = y_all[test_mask]
subjects_test = subjects_all[test_mask]
source_labels_test = source_labels_all[test_mask]

rest_mask = y_train_raw == 0
active_indices = np.flatnonzero(~rest_mask)
rest_indices = np.flatnonzero(rest_mask)

if rest_indices.shape[0] > REST_CAP:
    hardest_rest = rest_indices[np.argsort(rest_scores_train_raw[rest_indices])[-REST_CAP:]]
    keep_indices = np.sort(np.concatenate([active_indices, hardest_rest]))
else:
    keep_indices = np.arange(y_train_raw.shape[0])

X_train_raw = X_train_raw[keep_indices]
y_train = y_train_raw[keep_indices]
subjects_train = subjects_train_raw[keep_indices]
source_labels_train = source_labels_train_raw[keep_indices]


def normalize_by_subject(X, subjects):
    X_normalized = np.empty_like(X, dtype=np.float32)
    stats = {}
    for subject in sorted(np.unique(subjects)):
        mask = subjects == subject
        scaler = StandardScaler()
        X_subject = scaler.fit_transform(X[mask]).astype(np.float32)
        X_normalized[mask] = X_subject
        stats[str(subject)] = {
            'windows': int(mask.sum()),
            'mean_abs_after_normalization': float(np.mean(np.abs(np.mean(X_subject, axis=0)))),
            'mean_std_after_normalization': float(np.mean(np.std(X_subject, axis=0))),
        }
    return X_normalized, stats


X_train, train_normalization = normalize_by_subject(X_train_raw, subjects_train)
X_test, test_normalization = normalize_by_subject(X_test_raw, subjects_test)

print(f'X_train: {X_train.shape}')
print(f'X_test: {X_test.shape}')
print(f'Train counts: {Counter(y_train.tolist())}')
print(f'Test counts: {Counter(y_test.tolist())}')


## Step 7 - Save Artifacts


In [ ]:
split_name = 'holdout_' + '_'.join(TEST_SUBJECTS)
base_name = f'nano_myo_features_{split_name}_{FEATURE_SCHEMA_VERSION}'

paths = {
    'npz': FEATURE_DIR / f'{base_name}.npz',
    'metadata': FEATURE_DIR / f'{base_name}_metadata.json',
    'X_train': FEATURE_DIR / f'X_train_{FEATURE_SCHEMA_VERSION}.npy',
    'y_train': FEATURE_DIR / f'y_train_{FEATURE_SCHEMA_VERSION}.npy',
    'X_test': FEATURE_DIR / f'X_test_{FEATURE_SCHEMA_VERSION}.npy',
    'y_test': FEATURE_DIR / f'y_test_{FEATURE_SCHEMA_VERSION}.npy',
    'subjects_train': FEATURE_DIR / f'subjects_train_{FEATURE_SCHEMA_VERSION}.npy',
    'subjects_test': FEATURE_DIR / f'subjects_test_{FEATURE_SCHEMA_VERSION}.npy',
    'source_labels_train': FEATURE_DIR / f'source_labels_train_{FEATURE_SCHEMA_VERSION}.npy',
    'source_labels_test': FEATURE_DIR / f'source_labels_test_{FEATURE_SCHEMA_VERSION}.npy',
}

np.save(paths['X_train'], X_train)
np.save(paths['y_train'], y_train)
np.save(paths['X_test'], X_test)
np.save(paths['y_test'], y_test)
np.save(paths['subjects_train'], subjects_train)
np.save(paths['subjects_test'], subjects_test)
np.save(paths['source_labels_train'], source_labels_train)
np.save(paths['source_labels_test'], source_labels_test)

np.savez_compressed(
    paths['npz'],
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    subjects_train=subjects_train,
    subjects_test=subjects_test,
    source_labels_train=source_labels_train,
    source_labels_test=source_labels_test,
)

metadata = {
    'project': 'Project Nano-Myo',
    'feature_schema_version': FEATURE_SCHEMA_VERSION,
    'exercise_used': 'E2',
    'exercises_excluded': ['E1', 'E3'],
    'sample_rate_hz': SAMPLE_RATE_HZ,
    'window_ms': WINDOW_MS,
    'window_size_samples': WINDOW_SIZE,
    'overlap': OVERLAP,
    'step_size_samples': STEP_SIZE,
    'purity_threshold': PURITY_THRESHOLD,
    'feature_groups': FEATURE_GROUPS,
    'feature_names': FEATURE_NAMES,
    'input_dim': int(X_train.shape[1]),
    'train_subjects': TRAIN_SUBJECTS,
    'test_subjects': TEST_SUBJECTS,
    'e2_label_map': {str(key): int(value) for key, value in E2_LABEL_MAP.items()},
    'target_labels': {str(key): value for key, value in TARGET_LABELS.items()},
    'rest_cap': REST_CAP,
    'rest_windows_before_cap': int(rest_indices.shape[0]),
    'rest_windows_after_cap': int(np.sum(y_train == 0)),
    'train_class_counts': {str(key): int(value) for key, value in Counter(y_train.tolist()).items()},
    'test_class_counts': {str(key): int(value) for key, value in Counter(y_test.tolist()).items()},
    'train_source_label_counts': {str(key): int(value) for key, value in Counter(source_labels_train.tolist()).items()},
    'test_source_label_counts': {str(key): int(value) for key, value in Counter(source_labels_test.tolist()).items()},
    'normalization': {
        'method': 'per_subject_standardization',
        'fit_scope': 'each subject independently',
        'uses_labels': False,
        'train_subjects': train_normalization,
        'test_subjects': test_normalization,
    },
    'paths': {key: str(value) for key, value in paths.items()},
}

paths['metadata'].write_text(json.dumps(metadata, indent=2), encoding='utf-8')

for key, value in paths.items():
    print(f'{key}: {value}')


## Step 8 - Verify Artifacts


In [ ]:
with open(paths['metadata'], 'r', encoding='utf-8') as f:
    saved_metadata = json.load(f)

saved_X_train = np.load(paths['X_train'])
saved_y_train = np.load(paths['y_train'])
saved_X_test = np.load(paths['X_test'])
saved_y_test = np.load(paths['y_test'])

assert saved_metadata['feature_schema_version'] == FEATURE_SCHEMA_VERSION
assert saved_metadata['exercise_used'] == 'E2'
assert saved_metadata['input_dim'] == 80
assert 'wl' in saved_metadata['feature_groups']
assert 'ssc' in saved_metadata['feature_groups']
assert saved_X_train.shape[1] == 80
assert saved_X_test.shape[1] == 80
assert saved_X_train.shape[0] == saved_y_train.shape[0]
assert saved_X_test.shape[0] == saved_y_test.shape[0]
assert int(np.sum(saved_y_train == 0)) <= REST_CAP

print('Notebook 02 artifacts verified')
